# Running KBMOD with Noise-Added Images

We seek to gague KBMOD performance on images where there is more noise than usual for our brown dwarfs. We will do this by generating noisy images, running the KBMOD pipeline on said images, and obtaining the final returned likelihood value for our true brown dwarf trajectory.

In [9]:
import logging
logging.basicConfig(level=logging.INFO)
import pandas as pd
import os
from pathlib import Path
import shutil

from kbmod.image_collection import ImageCollection
from kbmod.standardizers.fits_standardizers.kbmodv05 import KBMODV0_5, KBMODV0_5Config
from kbmod.configuration import SearchConfiguration
from kbmod.run_search import SearchRunner

import astropy.units as u
from astropy.coordinates import SkyCoord
import numpy as np
from pathlib import Path
from astropy.io import fits
from astropy.wcs import WCS
from astropy.time import Time
from astropy.table import Table
from astropy.wcs.utils import skycoord_to_pixel
import pixel2icrs as p2i

In [10]:
def add_white_noise(input_dir, sigma):
    """
    Add white Gaussian noise to FITS files (SCIENCE, MASK, VARIANCE HDU structure).

    Parameters
    ----------
    input_dir : str
        Path to directory containing FITS files or subdirectories of FITS files.
    sigma : float
        Standard deviation of the white Gaussian noise.
    output_dir : str
        Path to directory where modified FITS files will be saved, preserving structure.
    """
    output_dir = Path("./i_band_{sigma}_sig")
    for root, dirs, files in os.walk(input_dir):
        # Preserve subdirectory structure in output_dir
        rel_path = os.path.relpath(root, input_dir)
        out_root = os.path.join(output_dir, rel_path)
        os.makedirs(out_root, exist_ok=True)

        for fname in files:
            if not fname.lower().endswith(".fits"):
                continue  # skip non-FITS files

            in_path = os.path.join(root, fname)
            out_path = os.path.join(out_root, fname)

            with fits.open(in_path) as hdul:
                # Copy to new HDU list
                new_hdul = fits.HDUList()

                # --- DEFAULT HDU (0) ---
                new_hdul.append(fits.ImageHDU(data=hdul[0].data, header=hdul[0].header))

                # --- SCIENCE HDU (1) ---
                sci = hdul[1].data.astype(float)
                noise = np.random.normal(0, sigma, size=sci.shape)
                sci_noisy = sci + noise
                new_hdul.append(fits.PrimaryHDU(data=sci_noisy, header=hdul[1].header))

                # --- MASK HDU (2) ---
                new_hdul.append(fits.ImageHDU(data=hdul[2].data, header=hdul[2].header))

                # --- VARIANCE HDU (3) ---
                var = hdul[3].data.astype(float)
                var_updated = var + sigma**2
                new_hdul.append(fits.ImageHDU(data=var_updated, header=hdul[3].header))

                # Write out new FITS file
                new_hdul.writeto(out_path, overwrite=True)
        return output_dir

In [11]:
ps1_bit_flag_map = {
    "DETECTOR": 2**0,
    "FLAT": 2**1,
    "DARK": 2**2,
    "BLANK": 2**3,
    "CTE": 2**4,
    "SAT": 2**5,
    "LOW": 2**6,
    "SUSPECT": 2**7,
    "BURNTOOL": 2**8,
    "CR": 2**9,
    "SPIKE": 2**10,
    "GHOST": 2**11,
    "STREAK": 2**12,
    "STARCORE": 2**13,
    "CONV.BAD": 2**14,
    "CONV.POOR": 2**15,
    "MARK": 2**16
}

ps1_mask_flags = ["DETECTOR", "BLANK", "CR", 
                  "SPIKE", "GHOST", "STARCORE",
                  "CONV.BAD", "STREAK", "BURNTOOL"]

raw_imgs_dir = Path("./kbmod_fits_examples/i_band")


In [ ]:
ra = 228.7472256453729 * u.deg
dec = 48.8013123316322 * u.deg
pmra = -938.043889 * u.mas / u.yr
pmdec = 1464.629578 * u.mas / u.yr
pmra_cos_dec = pmra * np.cos(dec.to(u.rad))
ref_epoch = Time("J2000")
hpms_coords = SkyCoord(ra=ra, dec=dec, 
                       pm_ra_cosdec=pmra_cos_dec, pm_dec=pmdec, 
                       frame='icrs', obstime=ref_epoch)

img_path = Path("./kbmod_fits_examples/i_band/rings.v3.skycell.2327.023.wrp.i.54985_42884.combined.fits")

with fits.open(img_path.open('rb')) as hdul:
    hdr = hdul[1].header
    wcs = WCS(hdr)
    
hpms_pixels = skycoord_to_pixel(hpms_coords, wcs)
print(f"Pixel Coords are: x={hpms_pixels[0]}, y={hpms_pixels[1]}, type={type(hpms_pixels[0])}")

t1 = ref_epoch + 1 * u.day
one_day_delta = hpms_coords.apply_space_motion(new_obstime=t1)
pixels_one_day_delta = skycoord_to_pixel(one_day_delta, wcs)

pm_pixels = np.array(pixels_one_day_delta) - np.array(hpms_pixels)

# Substitute known values here if you have them.
known_x = int(hpms_pixels[0])  # The object's x pixel coordinate at the first time.
known_x = 4707
known_y = int(hpms_pixels[1])  # The object's y pixel coordinate at the first time.
known_y = 446
known_vx = p2i.masyr2ppd(938.043889)  # The object's x velocity in pixels per day.
known_vy = p2i.masyr2ppd(1464.629578) # The object's y velocity in pixels per day.
print(f"vx: {known_vx}, vy:{known_vy}")

input_parameters = {
    "x_pixel_bounds": [known_x, known_x+1],
    "y_pixel_bounds": [known_y, known_y+1],

    # Use search parameters (including a force ecliptic angle of 0.0)
    # to match what we know is in the demo data.
    "generator_config": {
        "name": "VelocityGridSearch",
        "vx_steps": 100,
        "min_vx": known_vx * 0.5,
        "max_vx": known_vx * 2,
        "vy_steps": 100,
        "min_vy": known_vy * 0.5,
        "max_vy": known_vy * 2,
    },
    # Output parameters
    "result_filename": "./run_results/results_dist.ecsv",
    # Basic filtering (always applied)
    "num_obs": 0,  # <-- Filter anything with fewer than 15 observations
    "lh_level": 0.0,  # <-- Filter anything with a likelihood < 10.0
    # SigmaG clipping parameters
    "sigmaG_lims": [25, 75],  # <-- Clipping parameters (lower and upper percentile)
    # Other parameters
    "cpu_only": True,  # <-- This will be absurdly slow.  Set to False if you have a good enough GPU.
    "coadds": ["mean","median"],
    "save_all_stamps": True,
    #"near_dup_thresh":0,
    "results_per_pixel": 100**2,
    "do_clustering": True,
    "max_results":100**2
} 

Pixel Coords are: x=4713.048891188855, y=454.6687334780654, type=<class 'numpy.ndarray'>
vx: 0.010285706191308043, vy:0.01605974911735445


/Users/joaopassos/Software/HPMS_search/kbmod_env/lib/python3.11/site-packages/erfa/core.py:133: ErfaWarning: ERFA function "pmsafe" yielded 1 of "distance overridden (Note 6)"
  warn(f'ERFA function "{func_name}" yielded {wmsg}', ErfaWarning)


In [ ]:
def kbmod_pipeline(bit_flag_map, mask_flags, filepath, input_parameters, sigma):
    load_config = KBMODV0_5Config(mask_flags=mask_flags, bit_flag_map=bit_flag_map)
    ic = ImageCollection.fromDir(filepath, force=KBMODV0_5, config=load_config)
    wu = ic.toWorkUnit()

    input_parameters["result_filename"] = f"./run_results/results_dist_{sigma}_sigma.ecsv"

    config = SearchConfiguration.from_dict(input_parameters)
    wu.config = config
    rs = SearchRunner()
    results = rs.run_search_from_work_unit(wu)

    likelihood = results.table['likelihood'].max()

    return likelihood

def noise_runs(imgs_path, noise_min, noise_max, noise_steps,
               bit_flag_map, mask_flags, input_parameters):
    assert((noise_min < noise_max) and (noise_steps > 0))

    rows = []
    curr_noise = noise_min

    while (curr_noise < noise_max):
        noise_imgs_path = add_white_noise(imgs_path, sigma=curr_noise)
        likelihood = kbmod_pipeline(bit_flag_map, mask_flags, 
                                    noise_imgs_path, input_parameters, sigma)
        rows.append({"sigma": curr_noise, "likelihood": likelihood})
        # Delete generated images
        shutil.rmtree(noise_imgs_path)

        curr_noise += (noise_max-noise_min)/noise_steps

    df = pd.DataFrame(rows)
    return df


In [ ]:
noise_min = 1
noise_max = 20
noise_steps = 40

results = noise_runs(raw_imgs_dir, noise_min, noise_max, noise_steps, 
                     ps1_bit_flag_map, ps1_mask_flags, input_parameters)